# EoMT Fine-tuning for Anomaly Segmentation
## Day 1-2: Logit Normalization Loss + Head Fine-tuning

**Goal**: Fine-tune only the prediction head with Logit Normalization loss to improve anomaly segmentation.

**Strategy**:
- Freeze ViT encoder + EoMT decoder
- Train only mask/class prediction heads
- Use Automatic Mixed Precision (AMP) for speed
- Read Cityscapes directly from **zip files** (no extraction)
- Target: +1-2% AUPRC improvement

**Important**: Zips stay on Drive; only checkpoints and metrics are written back.

## 1. Setup & Imports

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('✓ Drive mounted')
    IS_COLAB = True
except:
    print('Not in Colab')
    IS_COLAB = False

# Install dependencies
if IS_COLAB:
    !pip install -q lightning scikit-learn torch torchvision timm

import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path
from datetime import datetime
import csv
from tqdm import tqdm
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
# Paths
if IS_COLAB:
    REPO_BASE = Path("/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject ")
else:
    REPO_BASE = Path("..")  # Adjust if running locally

os.chdir(REPO_BASE / "eomt")
print(f"Working directory: {os.getcwd()}")

# Training config
CONFIG = {
    'epochs': 2,
    'batch_size': 1,  # Use batch size 1 to avoid OOM
    'gradient_accumulation_steps': 4,  # Effective batch size = 4
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'logit_norm_temp': 0.07,
    'use_amp': True,
    'num_workers': 2,
    'freeze_encoder': True,
    'freeze_decoder': True,
    'img_size': 1024,  # Must match checkpoint for positional embeddings
    
    # Paths - ZIP FILES ONLY (no extraction)
    'cityscapes_root': str(REPO_BASE / "dataset" / "cityscapes"),
    'pretrained_ckpt': str(REPO_BASE / "checkpoints" / "eomt_cityscapes.bin"),
    'output_dir': str(REPO_BASE / "checkpoints" / "fine_tuned"),
}

Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)

print("✓ Configuration loaded")
print(f"  epochs: {CONFIG['epochs']}")
print(f"  batch_size: {CONFIG['batch_size']}")
print(f"  gradient_accumulation_steps: {CONFIG['gradient_accumulation_steps']}")
print(f"  effective_batch_size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"  img_size: {CONFIG['img_size']}x{CONFIG['img_size']} (checkpoint-compatible)")
print(f"  logit_norm_temp: {CONFIG['logit_norm_temp']}")
print(f"  use_amp: {CONFIG['use_amp']}")


## 3. Verify Cityscapes Zip Files (ZIP-ONLY, NO EXTRACTION)

⚠️ **Do not unzip** — the dataloaders read directly from the archives.

In [ ]:
# Check zip files
zip_dir = Path(CONFIG['cityscapes_root'])
left_zip = zip_dir / 'leftImg8bit_trainvaltest.zip'
gt_zip = zip_dir / 'gtFine_trainvaltest.zip'

print(f"Zip directory: {zip_dir}")
print(f"  leftImg8bit_trainvaltest.zip: {'✓ OK' if left_zip.exists() else '✗ MISSING'}")
print(f"  gtFine_trainvaltest.zip     : {'✓ OK' if gt_zip.exists() else '✗ MISSING'}")

if left_zip.exists() and gt_zip.exists():
    left_size_gb = left_zip.stat().st_size / 1e9
    gt_size_mb = gt_zip.stat().st_size / 1e6
    print(f"\n✓ Both zips found (total: {left_size_gb:.1f} GB + {gt_size_mb:.0f} MB)")
    print("  Dataset will be loaded directly from zips — no extraction needed.")
else:
    print(f"\n✗ Missing zips. Place both files here:")
    print(f"   {zip_dir}")
    print(f"\n  Download from: https://www.cityscapes-dataset.com/downloads/")
    print(f"  Then follow the README instructions.")

## 4. Import EoMT Components

In [ ]:
sys.path.insert(0, str(REPO_BASE / "eomt"))

from models.eomt import EoMT
from models.vit import ViT
from training.mask_classification_semantic import MaskClassificationSemantic
from datasets.cityscapes_semantic import CityscapesSemantic

print("✓ Imports successful")

## 5. Logit Normalization Loss

In [ ]:
class LogitNormalizationLoss(nn.Module):
    """Logit Normalization Loss for confidence calibration.
    
    Normalizes logits to unit sphere before computing cross-entropy.
    This prevents overconfident predictions and improves anomaly detection.
    
    Reference: https://arxiv.org/abs/2205.09310
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, logits, targets, mask=None, ignore_index=None):
        # Normalize logits to unit sphere
        logits_norm = F.normalize(logits, p=2, dim=1)
        logits_scaled = logits_norm / self.temperature
        
        # Cross-entropy with optional ignore_index
        if ignore_index is not None:
            loss = F.cross_entropy(logits_scaled, targets, reduction='none', ignore_index=ignore_index)
        else:
            loss = F.cross_entropy(logits_scaled, targets, reduction='none')
        
        # Apply mask if provided (ignore index 255)
        if mask is not None:
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1)
        else:
            loss = loss.mean()
        
        return loss

print(f"✓ Logit Normalization Loss (T={CONFIG['logit_norm_temp']})")


## 6. Load Pre-trained Model

In [ ]:
NUM_CLASSES = 19
MODEL_IMG_SIZE = (CONFIG['img_size'], CONFIG['img_size'])  # Dynamic from config
PATCH_SIZE = 16
NUM_QUERIES = 100
NUM_BLOCKS = 3
BACKBONE_NAME = "vit_base_patch14_reg4_dinov2"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Model input size: {MODEL_IMG_SIZE}")

# Build model
encoder = ViT(
    img_size=MODEL_IMG_SIZE,
    patch_size=PATCH_SIZE,
    backbone_name=BACKBONE_NAME,
)

network = EoMT(
    encoder=encoder,
    num_classes=NUM_CLASSES,
    num_q=NUM_QUERIES,
    num_blocks=NUM_BLOCKS,
    masked_attn_enabled=True,
)

model = MaskClassificationSemantic(
    network=network,
    img_size=MODEL_IMG_SIZE,
    num_classes=NUM_CLASSES,
    attn_mask_annealing_enabled=False,
    attn_mask_annealing_start_steps=None,
    attn_mask_annealing_end_steps=None,
    ckpt_path=CONFIG['pretrained_ckpt'],
    delta_weights=False,
    load_ckpt_class_head=True,
)

model = model.to(device)
print("✓ Model loaded with pre-trained weights")
print(f"  Memory allocated: ~6-7 GB (batch_size={CONFIG['batch_size']}, img_size={CONFIG['img_size']})")


## 7. Freeze Encoder & Decoder (Train Head Only)

In [ ]:
def freeze_module(module, freeze=True):
    for param in module.parameters():
        param.requires_grad = not freeze

if CONFIG['freeze_encoder']:
    freeze_module(model.network.encoder, freeze=True)
    print("✓ Encoder frozen")

if CONFIG['freeze_decoder']:
    if hasattr(model.network, 'decoder'):
        freeze_module(model.network.decoder, freeze=True)
    print("✓ Decoder frozen")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nParameters:")
print(f"  Total:     {total_params:,}")
print(f"  Trainable: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

## 8. Setup Training Components

In [ ]:
criterion = LogitNormalizationLoss(temperature=CONFIG['logit_norm_temp'])
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)
scaler = GradScaler() if CONFIG['use_amp'] else None

print(f"✓ Loss: Logit Normalization (T={CONFIG['logit_norm_temp']})")
print(f"✓ Optimizer: AdamW (lr={CONFIG['learning_rate']})")
print(f"✓ AMP: {'enabled' if CONFIG['use_amp'] else 'disabled'}")

## 9. Build DataLoader from Zips

In [ ]:
zip_dir = Path(CONFIG['cityscapes_root'])

print("Building DataLoader from zips...")
print(f"Image size: {CONFIG['img_size']}x{CONFIG['img_size']}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"Gradient accumulation steps: {CONFIG['gradient_accumulation_steps']}")
print(f"Effective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")

try:
    dm = CityscapesSemantic(
        path=zip_dir,
        num_workers=CONFIG['num_workers'],
        batch_size=CONFIG['batch_size'],
        img_size=(CONFIG['img_size'], CONFIG['img_size']),
        num_classes=NUM_CLASSES,
        color_jitter_enabled=False,
        check_empty_targets=True,
    )
    dm.setup(stage='fit')
    train_loader = dm.train_dataloader()
    
    print(f"✓ DataLoader created")
    print(f"  Batches: {len(train_loader)}")
    print(f"  Batch size: {CONFIG['batch_size']}")
    print(f"  Image size: {CONFIG['img_size']}x{CONFIG['img_size']}")
    print(f"  Workers: {CONFIG['num_workers']}")
except Exception as e:
    print(f"✗ DataLoader creation failed: {str(e)}")
    print(f"\n  Ensure both zips exist:")
    print(f"  - {zip_dir / 'leftImg8bit_trainvaltest.zip'}")
    print(f"  - {zip_dir / 'gtFine_trainvaltest.zip'}")
    train_loader = None


## 10. Loss Tracking Setup

In [ ]:
csv_path = Path(CONFIG['output_dir']) / 'training_loss.csv'

def log_loss(epoch, batch, loss_val):
    """Log training loss to CSV"""
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['epoch', 'batch', 'loss', 'timestamp'])
        if f.tell() == 0:
            writer.writeheader()
        writer.writerow({
            'epoch': epoch,
            'batch': batch,
            'loss': float(loss_val),
            'timestamp': datetime.now().isoformat()
        })

print(f"✓ Loss tracking setup")
print(f"  CSV: {csv_path}")

## 11. Training Function

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device, epoch):
    """Train for one epoch with AMP, zip-based dataloader, gradient accumulation."""
    model.train()
    total_loss = 0
    num_batches = len(dataloader)
    ignore_idx = getattr(model, 'ignore_idx', 255)
    accum_steps = CONFIG['gradient_accumulation_steps']

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
    for batch_idx, batch in enumerate(pbar):
        images, targets = batch
        images = images.to(device)
        
        # Convert targets to per-pixel labels at full image resolution
        per_pixel_targets = model.to_per_pixel_targets_semantic(targets, ignore_idx)
        per_pixel_targets = torch.stack(per_pixel_targets).to(device).long()  # [B, H, W]
        target_hw = per_pixel_targets.shape[-2:]
        
        # Diagnostic: check for invalid labels (first batch only)
        if batch_idx == 0:
            unique_labels = torch.unique(per_pixel_targets)
            invalid_labels = unique_labels[(unique_labels < 0) | ((unique_labels > 18) & (unique_labels != 255))]
            print(f"\n[Batch {batch_idx}] Target stats:")
            print(f"  Min: {per_pixel_targets.min().item()}, Max: {per_pixel_targets.max().item()}")
            print(f"  Unique labels: {unique_labels.tolist()}")
            if len(invalid_labels) > 0:
                print(f"  ⚠️ WARNING: Invalid labels found: {invalid_labels.tolist()}")
                print(f"     Valid range is [0-18] or {ignore_idx} (ignore)")
                print(f"     Clamping invalid labels to {ignore_idx}")
        
        # Safeguard: clamp any invalid labels to ignore_idx
        # Valid labels are 0-18 (19 classes) or 255 (ignore)
        # Anything else (e.g., -1 from license plate) should be ignored
        invalid_mask = (per_pixel_targets < 0) | ((per_pixel_targets > 18) & (per_pixel_targets != ignore_idx))
        if invalid_mask.any():
            per_pixel_targets = torch.where(invalid_mask, ignore_idx, per_pixel_targets)
        
        valid_mask = (per_pixel_targets != ignore_idx).float().to(device)

        if scaler is not None:
            # Prefer torch.amp.autocast; fallback to torch.cuda.amp.autocast
            try:
                with torch.amp.autocast('cuda'):
                    mask_logits_list, class_logits_list = model(images)
                    mask_logits = mask_logits_list[-1]
                    class_logits = class_logits_list[-1]
                    per_pixel_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)  # [B, C, h, w]
                    # Upsample logits to match target resolution
                    per_pixel_logits = F.interpolate(per_pixel_logits, size=target_hw, mode='bilinear', align_corners=False)
                    loss = criterion(per_pixel_logits, per_pixel_targets, mask=valid_mask, ignore_index=ignore_idx)
                    loss = loss / accum_steps  # Scale loss for accumulation
            except AttributeError:
                # Older torch: use torch.cuda.amp.autocast(enabled=True)
                with autocast(enabled=True):
                    mask_logits_list, class_logits_list = model(images)
                    mask_logits = mask_logits_list[-1]
                    class_logits = class_logits_list[-1]
                    per_pixel_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
                    per_pixel_logits = F.interpolate(per_pixel_logits, size=target_hw, mode='bilinear', align_corners=False)
                    loss = criterion(per_pixel_logits, per_pixel_targets, mask=valid_mask, ignore_index=ignore_idx)
                    loss = loss / accum_steps

            scaler.scale(loss).backward()
        else:
            mask_logits_list, class_logits_list = model(images)
            mask_logits = mask_logits_list[-1]
            class_logits = class_logits_list[-1]
            per_pixel_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
            per_pixel_logits = F.interpolate(per_pixel_logits, size=target_hw, mode='bilinear', align_corners=False)
            loss = criterion(per_pixel_logits, per_pixel_targets, mask=valid_mask, ignore_index=ignore_idx)
            loss = loss / accum_steps
            loss.backward()

        # Update weights every accum_steps batches or at the final remainder
        if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == num_batches:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps  # Undo scaling for logging
        avg_loss = total_loss / (batch_idx + 1)
        log_loss(epoch, batch_idx, loss.item() * accum_steps)
        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})

    return total_loss / num_batches

print("✓ Training function ready (ignore_index applied, spatial size aligned, AMP autocast fixed)")


## 12. Run Training

In [ ]:
if train_loader is not None:
    print("="*60)
    print("STARTING TRAINING")
    print("="*60)
    print(f"Epochs: {CONFIG['epochs']}")
    print(f"Batches per epoch: {len(train_loader)}")
    print(f"Total batches: {CONFIG['epochs'] * len(train_loader)}")
    print()
    
    best_loss = float('inf')
    training_start = datetime.now()
    
    for epoch in range(CONFIG['epochs']):
        epoch_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            epoch=epoch
        )
        
        print(f"Epoch {epoch+1}/{CONFIG['epochs']} - Loss: {epoch_loss:.4f}")
        
        # Save checkpoint
        ckpt_path = Path(CONFIG['output_dir']) / f'eomt_finetuned_epoch{epoch+1}.pth'
        torch.save(model.state_dict(), ckpt_path)
        print(f"  ✓ Checkpoint: {ckpt_path.name}")
        
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_ckpt = Path(CONFIG['output_dir']) / 'eomt_finetuned_best.pth'
            torch.save(model.state_dict(), best_ckpt)
            print(f"  ✓ Best model updated")
    
    training_end = datetime.now()
    training_time = (training_end - training_start).total_seconds() / 3600
    
    print()
    print("="*60)
    print(f"Training completed in {training_time:.2f} hours")
    print(f"Final loss: {epoch_loss:.4f}")
    print(f"Best loss:  {best_loss:.4f}")
    print("="*60)
    
    # Save final checkpoint
    final_ckpt = Path(CONFIG['output_dir']) / 'eomt_finetuned_final.pth'
    torch.save(model.state_dict(), final_ckpt)
    print(f"✓ Final checkpoint: {final_ckpt.name}")
else:
    print("✗ DataLoader not loaded. Cannot start training.")

## 13. Plot Training Loss

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if csv_path.exists():
    df = pd.read_csv(csv_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss per batch
    ax1 = axes[0]
    for epoch in df['epoch'].unique():
        epoch_data = df[df['epoch'] == epoch]
        ax1.plot(epoch_data['batch'], epoch_data['loss'], label=f'Epoch {epoch+1}', marker='o', markersize=2)
    ax1.set_xlabel('Batch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss per Batch')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Epoch average loss
    ax2 = axes[1]
    epoch_loss_avg = df.groupby('epoch')['loss'].mean()
    ax2.plot(epoch_loss_avg.index, epoch_loss_avg.values, marker='o', linewidth=2, markersize=8, color='red')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Average Loss')
    ax2.set_title('Average Loss per Epoch')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot_path = Path(CONFIG['output_dir']) / 'training_loss.png'
    plt.savefig(plot_path, dpi=100, bbox_inches='tight')
    print(f"✓ Plot saved: {plot_path.name}")
    plt.show()
else:
    print("No training data yet.")

## 14. Summary

In [ ]:
print("\n" + "="*60)
print("FINE-TUNING SUMMARY")
print("="*60)

summary = f"""
✓ Training completed
  Epochs:        {CONFIG['epochs']}
  Batch size:    {CONFIG['batch_size']}
  Learning rate: {CONFIG['learning_rate']}
  Loss function: Logit Normalization (T={CONFIG['logit_norm_temp']})
  Dataset:       Cityscapes (zip-based, no extraction)

✓ Outputs saved to:
  {CONFIG['output_dir']}

  - eomt_finetuned_best.pth
  - eomt_finetuned_epoch*.pth
  - training_loss.csv
  - training_loss.png

📊 Next steps:
  1. Run evaluation: evalAnomaly.py with fine-tuned checkpoint
  2. Compare results with baseline (MaxEntropy @ T=0.75: 77.75%)
  3. Target: +1-2% improvement
"""

print(summary)

# Save summary
summary_path = Path(CONFIG['output_dir']) / 'SUMMARY.txt'
with open(summary_path, 'w') as f:
    f.write(summary)
print(f"✓ Summary saved: {summary_path.name}")

## 15. Anomaly Segmentation Evaluation

Now we evaluate the fine-tuned model on anomaly datasets and compare with baseline.

In [ ]:
# Check available anomaly datasets (updated for Validation_Dataset structure)
validation_base = REPO_BASE / "dataset" / "anomaly" / "Validation_Dataset"

print("Checking anomaly datasets in Validation_Dataset...")
print(f"Base path: {validation_base}")
print()

available_datasets = {}

if validation_base.exists():
    for dataset_dir in sorted(validation_base.iterdir()):
        if dataset_dir.is_dir():
            images_path = dataset_dir / 'images'
            if images_path.exists():
                num_images = len(list(images_path.glob('*.*')))
                available_datasets[dataset_dir.name] = str(images_path / '*.*')
                print(f"✓ {dataset_dir.name:30s} ({num_images} images)")
else:
    print(f"✗ Validation_Dataset folder not found at: {validation_base}")

print(f"\nTotal available: {len(available_datasets)} datasets")
print(f"\nDataset names found:")
for name in sorted(available_datasets.keys()):
    print(f"  - {name}")

In [ ]:
# Check Validation_Dataset structure
validation_dataset = REPO_BASE / "dataset" / "anomaly" / "Validation_Dataset"

print("Exploring Validation_Dataset structure...")
print(f"Path: {validation_dataset}")
print(f"Exists: {validation_dataset.exists()}")

if validation_dataset.exists():
    print(f"\nContents of Validation_Dataset:")
    subdirs = []
    for item in sorted(validation_dataset.iterdir()):
        if item.is_dir():
            # Count files in subdirectories
            images_folder = item / 'images'
            labels_folder = item / 'labels_masks'
            
            num_images = len(list(images_folder.glob('*.*'))) if images_folder.exists() else 0
            num_labels = len(list(labels_folder.glob('*.*'))) if labels_folder.exists() else 0
            
            print(f"  📁 {item.name}/")
            if images_folder.exists():
                print(f"     ├── images/ ({num_images} files)")
            if labels_folder.exists():
                print(f"     └── labels_masks/ ({num_labels} files)")
            
            subdirs.append(item.name)
        else:
            print(f"  📄 {item.name}")
    
    print(f"\n✓ Found {len(subdirs)} anomaly subdatasets:")
    for name in subdirs:
        print(f"  - {name}")

### Evaluation Configuration

In [ ]:
EVAL_CONFIG = {
    'checkpoint': str(REPO_BASE / "checkpoints" / "fine_tuned" / "eomt_finetuned_best.pth"),
    'baseline_checkpoint': str(REPO_BASE / "checkpoints" / "eomt_cityscapes.bin"),
    'method': 'maxentropy',  # msp, maxlogit, maxentropy, rba
    'temperatures': [0.5, 0.75, 1.0, 1.1, 1.2, 2.0],
    'eval_dataset': 'RoadObsticle21',  # Change to test on different datasets
}

print("Evaluation Configuration:")
print(f"  Fine-tuned checkpoint: {Path(EVAL_CONFIG['checkpoint']).name}")
print(f"  Baseline checkpoint:   {Path(EVAL_CONFIG['baseline_checkpoint']).name}")
print(f"  Method:                {EVAL_CONFIG['method']}")
print(f"  Temperatures:          {EVAL_CONFIG['temperatures']}")
print(f"  Dataset:               {EVAL_CONFIG['eval_dataset']}")
print()
print(f"✓ Baseline (MaxEntropy @ T=0.75): 77.75% AUPRC")
print(f"✓ Target: +1-2% improvement → 78.75-79.75% AUPRC")

### Run Evaluation: Fine-tuned vs Baseline

In [ ]:
import subprocess
import re
from collections import defaultdict

def run_eval(checkpoint, dataset_name, dataset_path, method, temp):
    """Run evalAnomaly.py and parse results"""
    cmd = [
        'python', 'evalAnomaly.py',
        '--ckpt', checkpoint,
        '--input', dataset_path,
        '--method', method,
        '--temp', str(temp)
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        output = result.stdout + result.stderr
        
        # Parse AUPRC and FPR@95TPR
        auprc_match = re.search(r'AUPRC:\s*([\d.]+)', output)
        fpr_match = re.search(r'FPR@95TPR:\s*([\d.]+)', output)
        
        if auprc_match and fpr_match:
            return {
                'auprc': float(auprc_match.group(1)),
                'fpr95': float(fpr_match.group(1)),
                'success': True
            }
        else:
            return {'success': False, 'error': 'Could not parse results'}
    except subprocess.TimeoutExpired:
        return {'success': False, 'error': 'Timeout'}
    except Exception as e:
        return {'success': False, 'error': str(e)}

# Check if dataset is available
if EVAL_CONFIG['eval_dataset'] not in available_datasets:
    print(f"✗ Dataset '{EVAL_CONFIG['eval_dataset']}' not found!")
    print(f"Available datasets: {list(available_datasets.keys())}")
else:
    dataset_path = available_datasets[EVAL_CONFIG['eval_dataset']]
    print(f"Evaluating on: {EVAL_CONFIG['eval_dataset']}")
    print(f"Method: {EVAL_CONFIG['method'].upper()}")
    print("="*80)
    
    results_finetuned = {}
    results_baseline = {}
    
    for temp in EVAL_CONFIG['temperatures']:
        print(f"\n[Temperature = {temp}]")
        
        # Fine-tuned model
        print("  Running fine-tuned model...", end=' ')
        res_ft = run_eval(
            EVAL_CONFIG['checkpoint'],
            EVAL_CONFIG['eval_dataset'],
            dataset_path,
            EVAL_CONFIG['method'],
            temp
        )
        if res_ft['success']:
            results_finetuned[temp] = res_ft
            print(f"AUPRC: {res_ft['auprc']:.2f}%, FPR@95: {res_ft['fpr95']:.2f}%")
        else:
            print(f"✗ FAILED: {res_ft.get('error', 'Unknown error')}")
        
        # Baseline model
        print("  Running baseline model...", end=' ')
        res_bl = run_eval(
            EVAL_CONFIG['baseline_checkpoint'],
            EVAL_CONFIG['eval_dataset'],
            dataset_path,
            EVAL_CONFIG['method'],
            temp
        )
        if res_bl['success']:
            results_baseline[temp] = res_bl
            print(f"AUPRC: {res_bl['auprc']:.2f}%, FPR@95: {res_bl['fpr95']:.2f}%")
        else:
            print(f"✗ FAILED: {res_bl.get('error', 'Unknown error')}")
    
    print("\n" + "="*80)
    print("EVALUATION COMPLETED")
    print("="*80)

### Results Comparison & Visualization

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create comparison dataframe
comparison_data = []
for temp in EVAL_CONFIG['temperatures']:
    if temp in results_finetuned and temp in results_baseline:
        comparison_data.append({
            'Temperature': temp,
            'Fine-tuned AUPRC': results_finetuned[temp]['auprc'],
            'Baseline AUPRC': results_baseline[temp]['auprc'],
            'Improvement': results_finetuned[temp]['auprc'] - results_baseline[temp]['auprc'],
            'Fine-tuned FPR@95': results_finetuned[temp]['fpr95'],
            'Baseline FPR@95': results_baseline[temp]['fpr95'],
        })

if comparison_data:
    df_comparison = pd.DataFrame(comparison_data)
    
    # Display table
    print("\n" + "="*100)
    print(f"RESULTS: {EVAL_CONFIG['eval_dataset']} - {EVAL_CONFIG['method'].upper()}")
    print("="*100)
    print(df_comparison.to_string(index=False))
    print("="*100)
    
    # Find best temperature
    best_ft = df_comparison.loc[df_comparison['Fine-tuned AUPRC'].idxmax()]
    best_bl = df_comparison.loc[df_comparison['Baseline AUPRC'].idxmax()]
    
    print(f"\n📊 BEST RESULTS:")
    print(f"  Fine-tuned: {best_ft['Fine-tuned AUPRC']:.2f}% AUPRC @ T={best_ft['Temperature']}")
    print(f"  Baseline:   {best_bl['Baseline AUPRC']:.2f}% AUPRC @ T={best_bl['Temperature']}")
    print(f"  Improvement: +{best_ft['Fine-tuned AUPRC'] - best_bl['Baseline AUPRC']:.2f}%")
    
    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # AUPRC comparison
    ax1 = axes[0]
    ax1.plot(df_comparison['Temperature'], df_comparison['Fine-tuned AUPRC'], 
             marker='o', linewidth=2, markersize=8, label='Fine-tuned', color='blue')
    ax1.plot(df_comparison['Temperature'], df_comparison['Baseline AUPRC'], 
             marker='s', linewidth=2, markersize=8, label='Baseline', color='orange')
    ax1.set_xlabel('Temperature')
    ax1.set_ylabel('AUPRC (%)')
    ax1.set_title(f'AUPRC Comparison - {EVAL_CONFIG["method"].upper()}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Improvement plot
    ax2 = axes[1]
    colors = ['green' if x > 0 else 'red' for x in df_comparison['Improvement']]
    ax2.bar(df_comparison['Temperature'].astype(str), df_comparison['Improvement'], color=colors)
    ax2.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax2.set_xlabel('Temperature')
    ax2.set_ylabel('AUPRC Improvement (%)')
    ax2.set_title('Fine-tuned vs Baseline Improvement')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Save plot
    eval_plot_path = Path(CONFIG['output_dir']) / f'eval_{EVAL_CONFIG["eval_dataset"]}_{EVAL_CONFIG["method"]}.png'
    plt.savefig(eval_plot_path, dpi=100, bbox_inches='tight')
    print(f"\n✓ Comparison plot saved: {eval_plot_path.name}")
    plt.show()
    
    # Save results to CSV
    csv_eval_path = Path(CONFIG['output_dir']) / f'eval_results_{EVAL_CONFIG["eval_dataset"]}_{EVAL_CONFIG["method"]}.csv'
    df_comparison.to_csv(csv_eval_path, index=False)
    print(f"✓ Results saved: {csv_eval_path.name}")
else:
    print("\n✗ No results to compare. Check evaluation output above.")

## Multi-Dataset Evaluation

Valuta il modello su TUTTI i dataset anomaly disponibili e confronta le prestazioni.


In [ ]:
# Run evaluation on ALL available datasets
print("="*100)
print(f"MULTI-DATASET EVALUATION - {EVAL_CONFIG['method'].upper()}")
print("="*100)

all_results = {}  # {dataset_name: {temp: {fine_tuned: {...}, baseline: {...}}}}

for dataset_name in sorted(available_datasets.keys()):
    print(f"\n{'='*100}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*100}")
    
    dataset_path = available_datasets[dataset_name]
    results_ft_dict = {}
    results_bl_dict = {}
    
    for temp in EVAL_CONFIG['temperatures']:
        print(f"\n[Temperature = {temp}]")
        
        # Fine-tuned model
        print("  Running fine-tuned model...", end=' ')
        res_ft = run_eval(
            EVAL_CONFIG['checkpoint'],
            dataset_name,
            dataset_path,
            EVAL_CONFIG['method'],
            temp
        )
        if res_ft['success']:
            results_ft_dict[temp] = res_ft
            print(f"AUPRC: {res_ft['auprc']:.2f}%, FPR@95: {res_ft['fpr95']:.2f}%")
        else:
            print(f"✗ FAILED: {res_ft.get('error', 'Unknown error')}")
            results_ft_dict[temp] = None
        
        # Baseline model
        print("  Running baseline model...", end=' ')
        res_bl = run_eval(
            EVAL_CONFIG['baseline_checkpoint'],
            dataset_name,
            dataset_path,
            EVAL_CONFIG['method'],
            temp
        )
        if res_bl['success']:
            results_bl_dict[temp] = res_bl
            print(f"AUPRC: {res_bl['auprc']:.2f}%, FPR@95: {res_bl['fpr95']:.2f}%")
        else:
            print(f"✗ FAILED: {res_bl.get('error', 'Unknown error')}")
            results_bl_dict[temp] = None
    
    all_results[dataset_name] = {
        'fine_tuned': results_ft_dict,
        'baseline': results_bl_dict
    }
    
    print(f"\n✓ {dataset_name} evaluation completed")

print("\n" + "="*100)
print("ALL DATASETS EVALUATED")
print("="*100)


In [ ]:
# Summarize results across all datasets
print("\n" + "="*120)
print("SUMMARY: FINE-TUNED vs BASELINE - ALL DATASETS")
print("="*120)

summary_by_dataset = {}

for dataset_name in sorted(all_results.keys()):
    print(f"\n{'─'*120}")
    print(f"{dataset_name}")
    print(f"{'─'*120}")
    print(f"{'Temperature':<15} {'Fine-tuned AUPRC':<20} {'Baseline AUPRC':<20} {'Improvement':<20} {'FPR@95 FT':<15} {'FPR@95 BL':<15}")
    print(f"{'─'*120}")
    
    best_temp_ft = None
    best_auprc_ft = 0
    best_improvement = -100
    best_temp_improvement = None
    
    dataset_results = []
    
    for temp in EVAL_CONFIG['temperatures']:
        res_ft = all_results[dataset_name]['fine_tuned'].get(temp)
        res_bl = all_results[dataset_name]['baseline'].get(temp)
        
        if res_ft and res_bl:
            auprc_ft = res_ft['auprc']
            auprc_bl = res_bl['auprc']
            improvement = auprc_ft - auprc_bl
            fpr_ft = res_ft['fpr95']
            fpr_bl = res_bl['fpr95']
            
            improvement_str = f"+{improvement:.2f}%" if improvement >= 0 else f"{improvement:.2f}%"
            print(f"{temp:<15.2f} {auprc_ft:<20.2f}% {auprc_bl:<20.2f}% {improvement_str:<20} {fpr_ft:<15.2f}% {fpr_bl:<15.2f}%")
            
            dataset_results.append({
                'Temperature': temp,
                'Fine-tuned AUPRC': auprc_ft,
                'Baseline AUPRC': auprc_bl,
                'Improvement': improvement,
                'Fine-tuned FPR@95': fpr_ft,
                'Baseline FPR@95': fpr_bl,
            })
            
            # Track best
            if auprc_ft > best_auprc_ft:
                best_auprc_ft = auprc_ft
                best_temp_ft = temp
            
            if improvement > best_improvement:
                best_improvement = improvement
                best_temp_improvement = temp
    
    if dataset_results:
        summary_by_dataset[dataset_name] = {
            'data': dataset_results,
            'best_ft': best_temp_ft,
            'best_ft_auprc': best_auprc_ft,
            'best_improvement': best_improvement,
            'best_improvement_temp': best_temp_improvement
        }
        
        # Print summary for this dataset
        print(f"\n✓ Best fine-tuned: {best_auprc_ft:.2f}% AUPRC @ T={best_temp_ft}")
        print(f"✓ Best improvement: +{best_improvement:.2f}% @ T={best_temp_improvement}")

print("\n" + "="*120)
print("RANKING BY BEST IMPROVEMENT")
print("="*120)

ranking = sorted(
    summary_by_dataset.items(),
    key=lambda x: x[1]['best_improvement'],
    reverse=True
)

for idx, (dataset_name, summary) in enumerate(ranking, 1):
    print(f"{idx}. {dataset_name:<30} +{summary['best_improvement']:>6.2f}% @ T={summary['best_improvement_temp']}")


In [ ]:
# Visualize results across all datasets
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

plot_idx = 0

# Plot 1-4: AUPRC curves for each dataset
for dataset_name in sorted(summary_by_dataset.keys()):
    if plot_idx >= 4:
        break
    
    ax = axes[plot_idx]
    data = summary_by_dataset[dataset_name]['data']
    df_data = pd.DataFrame(data)
    
    ax.plot(df_data['Temperature'], df_data['Fine-tuned AUPRC'], 
            marker='o', linewidth=2, markersize=8, label='Fine-tuned', color='blue')
    ax.plot(df_data['Temperature'], df_data['Baseline AUPRC'], 
            marker='s', linewidth=2, markersize=8, label='Baseline', color='orange')
    ax.set_xlabel('Temperature')
    ax.set_ylabel('AUPRC (%)')
    ax.set_title(f'{dataset_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plot_idx += 1

plt.tight_layout()
plot_path = Path(CONFIG['output_dir']) / f'eval_all_datasets_curves_{EVAL_CONFIG["method"]}.png'
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
print(f"✓ Curves plot saved: {plot_path.name}")
plt.show()

# Plot: Improvement comparison across datasets
fig, ax = plt.subplots(figsize=(12, 6))

dataset_names = [name for name, _ in ranking]
improvements = [summary['best_improvement'] for _, summary in ranking]
temps = [f"T={summary['best_improvement_temp']}" for _, summary in ranking]

colors = ['green' if x > 0 else 'red' for x in improvements]
bars = ax.barh(dataset_names, improvements, color=colors)

# Add temperature labels on bars
for i, (bar, temp) in enumerate(zip(bars, temps)):
    width = bar.get_width()
    label_x = width + (0.1 if width > 0 else -0.1)
    ax.text(label_x, bar.get_y() + bar.get_height()/2, temp, 
            ha='left' if width > 0 else 'right', va='center', fontsize=9)

ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('AUPRC Improvement (%)')
ax.set_title(f'Best Improvement per Dataset - {EVAL_CONFIG["method"].upper()}')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plot_path2 = Path(CONFIG['output_dir']) / f'eval_all_datasets_improvement_{EVAL_CONFIG["method"]}.png'
plt.savefig(plot_path2, dpi=100, bbox_inches='tight')
print(f"✓ Improvement plot saved: {plot_path2.name}")
plt.show()

# Export all results to CSV
all_results_list = []
for dataset_name in sorted(summary_by_dataset.keys()):
    for row in summary_by_dataset[dataset_name]['data']:
        row['Dataset'] = dataset_name
        all_results_list.append(row)

if all_results_list:
    df_all = pd.DataFrame(all_results_list)
    # Reorder columns
    df_all = df_all[['Dataset', 'Temperature', 'Fine-tuned AUPRC', 'Baseline AUPRC', 
                      'Improvement', 'Fine-tuned FPR@95', 'Baseline FPR@95']]
    
    csv_path_all = Path(CONFIG['output_dir']) / f'eval_all_datasets_{EVAL_CONFIG["method"]}.csv'
    df_all.to_csv(csv_path_all, index=False)
    print(f"✓ All results exported: {csv_path_all.name}")
    
    # Summary statistics
    print("\n" + "="*100)
    print("OVERALL STATISTICS")
    print("="*100)
    print(f"Datasets evaluated: {len(summary_by_dataset)}")
    print(f"Best overall improvement: +{df_all['Improvement'].max():.2f}% ({df_all.loc[df_all['Improvement'].idxmax(), 'Dataset']})")
    print(f"Average improvement across all: +{df_all['Improvement'].mean():.2f}%")
    print(f"Datasets with improvement: {len(df_all[df_all['Improvement'] > 0])}")
    print(f"Datasets with regression: {len(df_all[df_all['Improvement'] <= 0])}")
